# Goldfarb-Iyengar Robust Portfolio Optimization — Empirical Backtest

This notebook implements a rigorous empirical backtest of the Goldfarb-Iyengar (2003) robust minimum-variance SOCP on S&P 500 data. The backtest tracks both portfolio performance and the Lagrangian dual variables, which provide economic interpretations developed in the accompanying paper.

**Structure:**
- Step 1 — Data loading and cleaning
- Step 2 — GI parameter estimation (rolling 60-month windows)
- Step 3 — SOCP implementation (CVXPY)
- Step 4 — Sanity checks on toy example
- Step 5 — Benchmark strategies
- Step 6 — Rolling backtest engine
- Step 7 — Performance metrics and dual variable analysis

---
## Step 1 — Data Loading and Cleaning

This section:
1. Parses the Bloomberg wide-format SPX total return index (TRI) data for ~503 current S&P 500 constituents
2. Converts daily TRI to monthly returns using the **correct formula for Bloomberg's inverted TRI convention**: `r_t = TRI_{t-1} / TRI_t - 1`
3. Loads and aligns Fama-French 5-factor + Momentum daily data, then compounds to monthly
4. Loads macro time series (VIX, MOVE, 3M T-bill)
5. Computes excess returns (stock returns minus RF)
6. Runs data quality checks and saves processed files to `data/processed/`

**Bloomberg TRI convention (important):** The `TOT_RETURN_INDEX_GROSS_DVDS` field exported via BDH for individual stocks is an *inverted* index — values *decrease* as the stock appreciates (e.g. AAPL: 377.5 in March 1991 → 0.33 in June 2026, representing a ~1135× total return). The SPX index TRI in `SPX_RETURN_DATA.csv` uses the standard convention and is handled separately.

**Known limitation:** The SPX data uses current constituents only (survivorship bias). WRDS data will be used later to correct for delisting bias.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_DIR = 'data/'
PROC_DIR = 'data/processed/'

print('Packages loaded.')

: 

### 1a. Load SPX Individual Stock TRI Data

The file `SPX_CUR_DATA.csv` is a Bloomberg BDH export. After inspection, it is a wide-format CSV with:
- Column 0: `Date` (daily, descending — most recent row first)
- Columns 1–503: one column per S&P 500 ticker, storing the total return index (`TOT_RETURN_INDEX_GROSS_DVDS`)

Stocks with shorter listing history have `NaN` in early rows.

In [ ]:
# ── Load raw SPX TRI data ──────────────────────────────────────────────────────
raw_spx = pd.read_csv(DATA_DIR + 'SPX_CUR_DATA.csv',
                      index_col=0, parse_dates=True)

# Data is stored descending (newest first); sort to ascending for time-series ops
raw_spx = raw_spx.sort_index()

# Strip Bloomberg exchange suffix so tickers are compact: 'AAPL UW Equity' → 'AAPL'
raw_spx.columns = raw_spx.columns.str.split(' ').str[0]

print(f'Raw SPX TRI: {raw_spx.shape[0]:,} daily rows × {raw_spx.shape[1]} stocks')
print(f'Date range : {raw_spx.index[0].date()} → {raw_spx.index[-1].date()}')
print()
print('First 5 rows (first 6 tickers):')
raw_spx.iloc[:5, :6]

In [ ]:
# ── Quality filter on TRI values ───────────────────────────────────────────────
# Replace zeros and negative TRI values with NaN — these are data errors.
# Bloomberg sometimes fills gaps with 0 or repeats a price, so also zero out
# runs of 5+ identical consecutive values (flat data = stale/errored feed).

tri_daily = raw_spx.copy().astype(float)
tri_daily[tri_daily <= 0] = np.nan

# Flag flat periods: if TRI unchanged for 5+ consecutive days, mask as NaN
def mask_flat_runs(series: pd.Series, min_run: int = 5) -> pd.Series:
    s = series.copy()
    # diff == 0 means unchanged; True where flat
    flat = s.diff().fillna(1) == 0
    run_len = flat.groupby((flat != flat.shift()).cumsum()).transform('sum')
    s[run_len >= min_run] = np.nan
    return s

tri_daily = tri_daily.apply(mask_flat_runs)

print(f'TRI after quality filter: {tri_daily.shape}')
print(f'NaN share: {tri_daily.isna().mean().mean():.1%}')

In [ ]:
# ── Resample to month-end and compute monthly returns ─────────────────────────
# Use last valid observation within each month as the month-end price.
tri_monthly = tri_daily.resample('ME').last()

# Bloomberg's TOT_RETURN_INDEX_GROSS_DVDS for individual stocks is an INVERTED index:
# values DECREASE as the stock appreciates. Verified empirically:
#   AAPL: 377.5 in Mar-1991 → 0.33 in Jun-2026  (≈1135× total return)
#   MSFT: 640.5 in Mar-1991 → 0.62 in Jun-2026  (≈1040× total return)
#
# Correct formula: r_t = TRI_{t-1} / TRI_t - 1
# (standard pct_change would give the wrong sign)
monthly_returns = tri_monthly.shift(1) / tri_monthly - 1

# Drop the first row (no prior month to form a return)
monthly_returns = monthly_returns.iloc[1:]

# Winsorise at 0.5th/99.5th percentile to remove residual data artefacts
# (e.g. a long NaN gap followed by a reconnecting TRI value → large spurious return)
lower = monthly_returns.stack().quantile(0.005)
upper = monthly_returns.stack().quantile(0.995)
monthly_returns = monthly_returns.clip(lower=lower, upper=upper)

print(f'Monthly returns: {monthly_returns.shape[0]} months × {monthly_returns.shape[1]} stocks')
print(f'Date range: {monthly_returns.index[0].date()} → {monthly_returns.index[-1].date()}')
print(f'Overall mean monthly return: {monthly_returns.stack().mean():.4f}  (should be ~0.005–0.010)')
print(f'Overall std monthly return:  {monthly_returns.stack().std():.4f}')

### 1b. Load Fama-French 5-Factor + Momentum Data

Ken French data files have multi-line text headers before the actual data. We parse them carefully and compound daily returns to monthly.

In [ ]:
# ── Parse FF5 daily ─────────────────────────────────────────────────────────
def load_ff_daily(path: str, skiprows: int) -> pd.DataFrame:
    """Load a Ken French daily factor CSV, convert percent→decimal, parse dates."""
    df = pd.read_csv(path, skiprows=skiprows, index_col=0, parse_dates=True)
    # Drop the copyright footer row (non-parseable date becomes NaT or string index)
    df = df[pd.to_datetime(df.index, errors='coerce').notna()]
    df.index = pd.to_datetime(df.index)
    df.index.name = 'Date'
    df = df / 100.0  # percent → decimal
    df.columns = df.columns.str.strip()
    return df

ff5_daily  = load_ff_daily(DATA_DIR + 'F-F_Research_Data_5_Factors_2x3_daily.csv', skiprows=3)
mom_daily  = load_ff_daily(DATA_DIR + 'F-F_Momentum_Factor_daily.csv', skiprows=13)
mom_daily.columns = ['MOM']

print(f'FF5  daily: {ff5_daily.shape}  | {ff5_daily.index[0].date()} → {ff5_daily.index[-1].date()}')
print(f'MOM  daily: {mom_daily.shape}  | {mom_daily.index[0].date()} → {mom_daily.index[-1].date()}')
print()
print('FF5 sample:')
ff5_daily.tail(3)

In [ ]:
# ── Compound daily factors to monthly ─────────────────────────────────────────
def compound_to_monthly(daily_df: pd.DataFrame) -> pd.DataFrame:
    """Compound daily factor returns (in decimal) to monthly."""
    return daily_df.resample('ME').apply(lambda col: (1 + col).prod() - 1)

ff5_monthly  = compound_to_monthly(ff5_daily)
mom_monthly  = compound_to_monthly(mom_daily)

# Combine into one factor dataframe: [MktRF, SMB, HML, RMW, CMA, MOM, RF]
factors = ff5_monthly[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']].copy()
factors = factors.rename(columns={'Mkt-RF': 'MktRF'})
factors['MOM'] = mom_monthly['MOM']
factors = factors.sort_index()

print(f'Monthly factors: {factors.shape}  | {factors.index[0].date()} → {factors.index[-1].date()}')
print()
factors.tail(3)

### 1c. Load Macro Data (VIX, MOVE, 3M T-bill)

In [ ]:
def load_bloomberg_daily(path: str, value_col: str = 'PX_LAST') -> pd.Series:
    """Load a Bloomberg daily price file (Date, PX_LAST, ...) as a Series."""
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df = df.sort_index()
    return df[value_col].rename(path.split('/')[-1].replace('_DATA.csv', ''))

vix_daily  = load_bloomberg_daily(DATA_DIR + 'VIX_DATA.csv')
move_daily = load_bloomberg_daily(DATA_DIR + 'MOVE_DATA.csv')
gb3_daily  = load_bloomberg_daily(DATA_DIR + 'GB3GVT_DATA.csv')  # 3M T-bill yield (%)
spx_tri    = load_bloomberg_daily(DATA_DIR + 'SPX_RETURN_DATA.csv',
                                  value_col='TOT_RETURN_INDEX_GROSS_DVDS')

macro_daily = pd.DataFrame({'VIX': vix_daily, 'MOVE': move_daily, 'GB3': gb3_daily})
macro_daily = macro_daily.sort_index()

# SPX benchmark: monthly total return
spx_monthly_ret = spx_tri.resample('ME').last().pct_change().iloc[1:]
spx_monthly_ret.name = 'SPX'

print(f'Macro daily: {macro_daily.shape}  | {macro_daily.index[0].date()} → {macro_daily.index[-1].date()}')
print(f'SPX monthly: {spx_monthly_ret.shape[0]} months')
macro_daily.tail(3)

### 1d. Align Data and Compute Excess Returns

We align the stock returns and factor returns to a common monthly date index, then subtract the risk-free rate to get excess returns used in factor regressions.

In [ ]:
# ── Find common date range ─────────────────────────────────────────────────────
# SPX: up to last available month
# Factors: FF5 daily data ends April 2026 → monthly ends April 2026
# We'll use intersection of both
common_start = max(monthly_returns.index[0], factors.index[0])
common_end   = min(monthly_returns.index[-1], factors.index[-1])

monthly_returns = monthly_returns.loc[common_start:common_end]
factors         = factors.loc[common_start:common_end]
spx_monthly_ret = spx_monthly_ret.reindex(monthly_returns.index)

print(f'Aligned date range: {common_start.date()} → {common_end.date()}')
print(f'Months: {len(monthly_returns)}')
print(f'Stocks: {monthly_returns.shape[1]}')
print()

# ── Excess returns ─────────────────────────────────────────────────────────────
# Subtract monthly RF from each stock's return
rf_monthly = factors['RF']  # already in decimal, compounded from daily
monthly_excess = monthly_returns.subtract(rf_monthly, axis=0)

print(f'Mean RF (monthly): {rf_monthly.mean():.4f} = {rf_monthly.mean()*12:.2%} annualised')
print(f'Monthly excess return mean: {monthly_excess.stack().mean():.4f}')

### 1e. Data Quality Checks

In [ ]:
# ── Missing data summary ───────────────────────────────────────────────────────
pct_missing = monthly_returns.isna().mean().sort_values(ascending=False)
n_stocks_total = monthly_returns.shape[1]

print(f'Total stocks: {n_stocks_total}')
print(f'Date range: {monthly_returns.index[0].date()} → {monthly_returns.index[-1].date()}')
print(f'Months: {len(monthly_returns)}')
print()
print('Missing data distribution:')
bins = [0, 0.01, 0.1, 0.25, 0.5, 1.0]
labels = ['<1%', '1–10%', '10–25%', '25–50%', '>50%']
for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:])):
    n = ((pct_missing > lo) & (pct_missing <= hi)).sum()
    print(f'  {labels[i]:>8}: {n:3d} stocks')

print()
print('Top 10 stocks with most missing data:')
print(pct_missing.head(10).to_string())

In [ ]:
# ── Average monthly return sanity check ───────────────────────────────────────
cross_mean = monthly_returns.mean(axis=1)  # equal-weight average across stocks
ann_mean = (1 + cross_mean).prod() ** (12 / len(cross_mean)) - 1
ann_vol  = cross_mean.std() * np.sqrt(12)

print(f'Equal-weight portfolio (cross-sectional mean):')
print(f'  Ann. return (geom): {ann_mean:.2%}')
print(f'  Ann. volatility:    {ann_vol:.2%}')
print(f'  Sharpe (approx):    {ann_mean/ann_vol:.2f}')
print()

# Return distribution sanity
stack = monthly_returns.stack()
print(f'Return distribution (all stocks, all months):')
print(f'  Min:    {stack.min():.4f}')
print(f'  5th pct:{stack.quantile(0.05):.4f}')
print(f'  Mean:   {stack.mean():.4f}')
print(f'  95th pct:{stack.quantile(0.95):.4f}')
print(f'  Max:    {stack.max():.4f}')

In [ ]:
# ── Plot 1: Number of stocks with valid returns per month ─────────────────────
n_valid = monthly_returns.notna().sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.fill_between(n_valid.index, n_valid.values, alpha=0.6, color='steelblue')
ax.axhline(503, color='red', ls='--', lw=1, label='503 (full universe)')
ax.set_title('Stocks with valid data per month (survivorship-biased universe)')
ax.set_xlabel('Date')
ax.set_ylabel('Number of stocks')
ax.set_ylim(0, 530)
ax.legend()

# ── Plot 2: SPX cumulative return vs equal-weight portfolio ───────────────────
ax2 = axes[1]
ew_cum  = (1 + cross_mean).cumprod()
spx_cum = (1 + spx_monthly_ret.reindex(monthly_returns.index)).cumprod()

ax2.plot(ew_cum.index, ew_cum.values, label='Equal-Weight (cross-section avg)', color='steelblue')
ax2.plot(spx_cum.index, spx_cum.values, label='SPX Total Return', color='orange', ls='--')
ax2.set_title('Cumulative return: equal-weight vs SPX')
ax2.set_xlabel('Date')
ax2.set_ylabel('Growth of $1')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Min stocks in any month: {n_valid.min()} ({n_valid.idxmin().date()})')
print(f'Max stocks in any month: {n_valid.max()}')

In [ ]:
# ── Plot 3: Factor return time series ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
factor_cols = ['MktRF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']
for col in factor_cols:
    cum = (1 + factors[col]).cumprod()
    ax.plot(cum.index, cum.values, label=col, lw=1)
ax.set_title('Cumulative Fama-French 5 + Momentum factor returns (monthly compounded)')
ax.set_ylabel('Growth of $1')
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

print('\nFactor summary statistics (monthly, decimal):')
print(factors[factor_cols].describe().round(4).T[['mean','std','min','max']])

In [ ]:
# ── Cross-sectional correlation check ─────────────────────────────────────────
# Use last 60 months, all stocks with ≥55 valid observations.
# NOTE: mean pairwise correlation across 478 diverse S&P 500 stocks is naturally low
# (~0.02–0.05) because inter-sector pairs (e.g. tech vs energy) drive the average.
# Within-sector pairs (AAPL–MSFT, GOOGL–META) are much higher (0.3–0.6).
recent = monthly_returns.iloc[-60:]
well_covered = recent.columns[recent.notna().sum() >= 55]
corr_full = recent[well_covered].corr()
n_full = len(well_covered)
corr_vals = corr_full.values[np.triu_indices(n_full, k=1)]

print(f'Stocks with ≥55 valid obs in last 60m: {n_full}')
print(f'All-pair XS correlation ({n_full} stocks):')
print(f'  Mean:   {corr_vals.mean():.3f}')
print(f'  Median: {np.median(corr_vals):.3f}')
print(f'  Std:    {corr_vals.std():.3f}')
print(f'  Min:    {corr_vals.min():.3f}   Max: {corr_vals.max():.3f}')
print()
print('Selected pairs (reality check):')
for a, b in [('AAPL','MSFT'), ('AAPL','JNJ'), ('AAPL','XOM'), ('MSFT','GOOGL')]:
    if a in well_covered and b in well_covered:
        c = recent[a].corr(recent[b])
        print(f'  {a} vs {b}: {c:.3f}')
print()
print('Monthly mean return by decade (survivorship bias inflates all):')
for start, end in [('1991','1999'), ('2000','2009'), ('2010','2019'), ('2020','2026')]:
    sub = monthly_returns.loc[start:end].stack()
    print(f'  {start}–{end}: mean={sub.mean():.4f}/month, std={sub.std():.3f}, n={len(sub):,}')

### 1f. Save Processed Data

In [ ]:
# ── Save to parquet ────────────────────────────────────────────────────────────
monthly_returns.to_parquet(PROC_DIR + 'monthly_returns.parquet')
monthly_excess.to_parquet(PROC_DIR + 'monthly_excess_returns.parquet')
factors.to_parquet(PROC_DIR + 'monthly_factors.parquet')
macro_daily.to_parquet(PROC_DIR + 'macro_daily.parquet')
spx_monthly_ret.to_frame().to_parquet(PROC_DIR + 'spx_benchmark.parquet')

print('Saved:')
print(f'  monthly_returns.parquet        → {monthly_returns.shape}')
print(f'  monthly_excess_returns.parquet → {monthly_excess.shape}')
print(f'  monthly_factors.parquet        → {factors.shape}')
print(f'  macro_daily.parquet            → {macro_daily.shape}')
print(f'  spx_benchmark.parquet          → {spx_monthly_ret.shape}')

### Step 1 Summary

| Item | Value |
|------|-------|
| Stocks | ~503 (current S&P 500 constituents) |
| Date range | 1990-01 → 2026-04 |
| Monthly observations | ~436 months |
| Factors | MktRF, SMB, HML, RMW, CMA, MOM + RF |
| Macro series | VIX, MOVE, 3M T-bill (daily) |

**Known issues:**
- Survivorship bias (current constituents only; historical membership not used)
- Newer IPOs (ABNB, DASH, etc.) have short histories — will be excluded from early windows by the 30-observation filter in Step 2
- FF5 daily data ends April 2026; returns beyond that will not have factor data

→ Proceed to Step 2 once quality checks above pass.

---
## Step 2 — GI Parameter Estimation

For each rolling 60-month estimation window ending at date `t`, we estimate the GI factor model parameters for each stock `i`:

$$r_{it} = \mu_i + V_{1i} f_{1t} + \cdots + V_{mi} f_{mt} + \varepsilon_{it}$$

from which we extract:
- `mu0`: estimated mean excess returns
- `V0 (m×n)`: factor loadings
- `F (m×m)`: factor covariance (sample)
- `G (m×m)`: GI precision matrix (equation 57)
- `D_bar (n,)`: idiosyncratic variance upper bounds
- `rho (n,)`: factor loading uncertainty radii (equation 57)
- `gamma (n,)`: mean return uncertainty radii (equation 56)

In [ ]:
from scipy import stats
from numpy.linalg import inv, eigvalsh
from typing import Optional


def estimate_gi_params(
    excess_returns_window: pd.DataFrame,
    factor_returns_window: pd.DataFrame,
    omega: float = 0.95,
    min_obs: int = 30,
) -> dict:
    """
    Estimate GI (2003) Section 5 parameters for one estimation window.

    G is the factor Gram matrix F^T M F (demeaned), which defines the
    factor-loading uncertainty set ||G^{1/2}(v - v_hat)||_2 <= rho_i.
    With this choice H = G^{-1/2} Sigma_F G^{-1/2} ≈ I_m / T so
    theta_max ≈ 1/T and sigma_max = T (a.k.a. the sample size).
    """
    T, n = excess_returns_window.shape
    m = factor_returns_window.shape[1]
    tickers = excess_returns_window.columns.tolist()

    ones  = np.ones((T, 1))
    F_mat = factor_returns_window.values         # (T, m)
    A     = np.hstack([ones, F_mat])             # (T, m+1) design matrix
    F_cov = np.cov(F_mat.T)                      # (m, m) factor covariance

    # G = F^T M F (Gram matrix of demeaned factors, m×m)
    B          = F_mat.T                         # (m, T)
    BBT        = B @ B.T                         # sum_t f_t f_t^T
    correction = (1.0 / T) * (B @ ones) @ (ones.T @ B.T)
    G = BBT - correction + 1e-8 * np.eye(m)     # regularised Gram matrix

    # F-distribution critical values
    df2   = T - m - 1
    c_v   = stats.f.ppf(omega, m, df2)          # m df — for v_i loadings only
    c_a   = stats.f.ppf(omega, 1, df2)          # 1 df — for alpha_i alone

    mu0   = np.full(n, np.nan)
    V0    = np.full((m, n), np.nan)
    s2    = np.full(n, np.nan)
    rho   = np.full(n, np.nan)
    gamma = np.full(n, np.nan)
    D_bar = np.full(n, np.nan)
    n_obs = np.zeros(n, dtype=int)

    for i, col in enumerate(tickers):
        y_full   = excess_returns_window[col].values
        mask     = ~np.isnan(y_full)
        n_obs[i] = mask.sum()
        if n_obs[i] < min_obs:
            continue
        y_i     = y_full[mask]
        A_i     = A[mask]
        T_i     = n_obs[i]
        df2_i   = T_i - m - 1
        if df2_i <= 0:
            continue

        ATA_i = A_i.T @ A_i + 1e-10 * np.eye(m + 1)
        x_hat = np.linalg.solve(ATA_i, A_i.T @ y_i)
        resid = y_i - A_i @ x_hat
        s2_i  = resid @ resid / df2_i

        mu0[i]   = x_hat[0]
        V0[:, i] = x_hat[1:]
        s2[i]    = s2_i
        D_bar[i] = s2_i

        # GI rho_i: radius of ||G^{1/2}(v - v_hat)||_2 uncertainty set
        # From F-test on v_i alone (m df): rho_i^2 = m * F_{m, T-m-1}(omega) * s2_i
        rho[i] = np.sqrt(m * c_v * s2_i)

        # GI gamma_i: half-width of confidence interval for alpha_i
        ATA_inv_i = inv(ATA_i)
        gamma[i]  = np.sqrt(ATA_inv_i[0, 0] * c_a * s2_i)

    return {
        'mu0'        : mu0,
        'V0'         : V0,
        'G'          : G,
        'F_cov'      : F_cov,
        'D_bar'      : D_bar,
        'rho'        : rho,
        'gamma'      : gamma,
        's2'         : s2,
        'valid_mask' : ~np.isnan(mu0),
        'ticker_names': tickers,
        'n_obs'      : n_obs,
    }


print('estimate_gi_params() defined (G = Gram matrix, rho uses m df).')

In [ ]:
# ── Quick test: estimate on the first 60-month window ─────────────────────────
# Reload processed data (in case notebook is run from scratch)
monthly_excess = pd.read_parquet(PROC_DIR + 'monthly_excess_returns.parquet')
factors        = pd.read_parquet(PROC_DIR + 'monthly_factors.parquet')

WINDOW = 60
factor_cols_regression = ['MktRF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']

exc_w0  = monthly_excess.iloc[:WINDOW]
fac_w0  = factors[factor_cols_regression].iloc[:WINDOW]

params0 = estimate_gi_params(exc_w0, fac_w0, omega=0.95)

n_valid = params0['valid_mask'].sum()
print(f'Window 0 (first {WINDOW} months): {n_valid} valid stocks')
print(f'mu0 range: [{np.nanmin(params0["mu0"]):.4f}, {np.nanmax(params0["mu0"]):.4f}]')
print(f'rho range: [{np.nanmin(params0["rho"]):.4f}, {np.nanmax(params0["rho"]):.4f}]')
print(f'gamma range: [{np.nanmin(params0["gamma"]):.4f}, {np.nanmax(params0["gamma"]):.4f}]')
print(f'D_bar range: [{np.nanmin(params0["D_bar"]):.6f}, {np.nanmax(params0["D_bar"]):.6f}]')
print()
print(f'G matrix (6×6), condition number: {np.linalg.cond(params0["G"]):.1f}')
print(f'F_cov eigenvalues: {eigvalsh(params0["F_cov"])}')

---
## Step 3 — SOCP Solver (GI Equation 32)

Implements the Goldfarb-Iyengar robust minimum-variance SOCP using CVXPY. After solving, we extract and verify all dual variables.

The primal problem is:
$$\min_{\phi, \lambda, \delta, \zeta, \sigma, \tau, t} \; \lambda + \delta$$

subject to constraints C1–C9 as derived in the paper. See the markdown cell in Step 4 for the full constraint listing.

In [ ]:
import cvxpy as cp
import time
from numpy.linalg import eigh, inv


def precompute_spectral(
    G: np.ndarray,
    F_cov: np.ndarray,
    V0_valid: np.ndarray,
) -> tuple:
    """
    H = G^{-1/2} F_cov G^{-1/2}, H = Q Λ Q^T
    A_mat = Q^T H^{1/2} G^{1/2} V0   (m × n_valid)

    With G = Gram matrix: H ≈ I_m/T so all theta_j ≈ 1/T and
    sigma_max = T (the window length).
    """
    m = G.shape[0]
    egvals_G, egvecs_G = eigh(G)
    egvals_G   = np.clip(egvals_G, 1e-12, None)
    G_half     = egvecs_G @ np.diag(np.sqrt(egvals_G)) @ egvecs_G.T
    G_inv_half = egvecs_G @ np.diag(1.0 / np.sqrt(egvals_G)) @ egvecs_G.T
    H          = G_inv_half @ F_cov @ G_inv_half
    H          = (H + H.T) / 2
    theta_raw, Q = eigh(H)
    theta = np.clip(theta_raw, 0, None)
    idx   = np.argsort(theta); theta = theta[idx]; Q = Q[:, idx]
    H_half = Q @ np.diag(np.sqrt(theta)) @ Q.T
    A_mat  = Q.T @ H_half @ G_half @ V0_valid
    return theta, Q, A_mat, theta[-1]


def solve_gi_socp(
    mu0_valid: np.ndarray,
    V0_valid: np.ndarray,
    G: np.ndarray,
    F_cov: np.ndarray,
    D_bar_valid: np.ndarray,
    rho_valid: np.ndarray,
    gamma_valid: np.ndarray,
    alpha: Optional[float] = None,   # None → pure min-var (no return floor)
    solver: str = 'CLARABEL',
    verbose: bool = False,
) -> dict:
    """
    Solve the GI (2003) robust minimum-variance SOCP (equation 32).

    alpha=None  → omit the return-floor constraint (C2) entirely — use this for
                  pure minimum-variance optimisation.
    alpha=<val> → worst-case portfolio excess return ≥ alpha (monthly decimal).

    GI dual variables are extracted as:
      mu2   : shadow price on C2 (0 when alpha=None)
      mu7   : shadow price on C7 (sigma ≤ 1/theta_max)
      mu8   = 1/sigma_opt          (from GI complementary slackness)
      mu9i  = 1/(1 - sigma*theta_j) (S-procedure multiplier per factor)
    """
    n = len(mu0_valid); m = G.shape[0]
    theta, Q, A_mat, theta_max = precompute_spectral(G, F_cov, V0_valid)
    if theta_max <= 0:
        return {'status': 'infeasible_theta', 'weights': None, 'dual_vars': None,
                'objective': np.nan, 'solve_time': 0.0}

    phi   = cp.Variable(n, nonneg=True)
    lam   = cp.Variable()
    delta = cp.Variable()
    zeta  = cp.Variable(n, nonneg=True)
    sigma = cp.Variable(nonneg=True)
    tau   = cp.Variable(nonneg=True)
    t     = cp.Variable(m, nonneg=True)

    w = A_mat @ phi
    r = rho_valid @ zeta
    D_half_phi = cp.multiply(np.sqrt(D_bar_valid), phi)

    C1  = cp.norm(cp.hstack([2 * D_half_phi, 1 - delta]), 2) <= 1 + delta
    C3  = phi <= zeta
    C4  = -phi <= zeta
    C5  = cp.sum(phi) == 1
    C6  = tau + cp.sum(t) <= lam
    C7  = sigma <= 1.0 / theta_max
    C8  = cp.norm(cp.hstack([2 * r, sigma - tau]), 2) <= sigma + tau
    C9_list = []
    for j in range(m):
        lhs = 1 - theta[j] * sigma - t[j]
        rhs = 1 - theta[j] * sigma + t[j]
        C9_list.append(cp.norm(cp.hstack([2 * w[j], lhs]), 2) <= rhs)

    constraints = [C1, C3, C4, C5, C6, C7, C8] + C9_list

    # C2: worst-case return floor (optional)
    C2 = None
    if alpha is not None:
        C2 = (mu0_valid - gamma_valid) @ phi >= alpha
        constraints.append(C2)

    objective = cp.Minimize(lam + delta)
    problem   = cp.Problem(objective, constraints)

    t_start = time.time()
    try:
        problem.solve(solver=solver, verbose=verbose)
    except Exception:
        try:
            problem.solve(solver=cp.CLARABEL, verbose=verbose)
        except Exception:
            problem.solve(solver=cp.ECOS, verbose=verbose)
    solve_time = time.time() - t_start

    if problem.status not in ['optimal', 'optimal_inaccurate']:
        return {'status': problem.status, 'weights': None, 'dual_vars': None,
                'objective': np.nan, 'solve_time': solve_time}

    def scalar_dual(c):
        if c is None: return np.nan
        v = c.dual_value
        return float(np.squeeze(v)) if v is not None else np.nan

    def vec_dual(c, size):
        if c is None: return np.full(size, np.nan)
        v = c.dual_value
        return np.array(v).flatten() if v is not None else np.full(size, np.nan)

    sigma_opt = float(sigma.value) if sigma.value is not None else np.nan
    tau_opt   = float(tau.value)   if tau.value   is not None else np.nan

    # GI dual variables (analytically derived from complementary slackness)
    mu8_gi  = 1.0 / sigma_opt if sigma_opt > 1e-10 else np.nan
    mu9i_gi = np.array([1.0 / (1.0 - sigma_opt * theta[j])
                         for j in range(m)]) if sigma_opt > 1e-10 else np.full(m, np.nan)

    return {
        'weights'   : phi.value,
        'objective' : float(problem.value),
        'dual_vars' : {
            'mu2'      : scalar_dual(C2),    # 0 when alpha=None
            'nu'       : scalar_dual(C5),    # budget constraint multiplier
            'mu7'      : scalar_dual(C7),    # S-procedure validity shadow price
            'mu8'      : mu8_gi,             # = 1/sigma_opt
            'mu9i'     : mu9i_gi,            # = 1/(1-sigma*theta_j), all equal when G=Gram
            'pi'       : vec_dual(C3, n),
            'qi'       : vec_dual(C4, n),
            'sigma_opt': sigma_opt,
            'tau_opt'  : tau_opt,
        },
        'status'    : problem.status,
        'solve_time': solve_time,
        'theta'     : theta,
        'theta_max' : theta_max,
    }


print('solve_gi_socp() defined (alpha=None for pure min-var, GI duals analytical).')

---
## Step 4 — Sanity Checks on Toy Example

Before the full backtest we solve the SOCP on a small 20-stock universe (first 60 months) and verify:

**Primal feasibility:**
- `weights.sum() ≈ 1.0`
- `all(weights >= 0)`
- `(mu0 - gamma) @ weights >= alpha`

**Dual variable signs:**
- `mu2 >= 0`
- `mu7 >= 0`
- `all(mu9i >= 1)` (since `1/(1-σθ_i) ≥ 1` when `σ, θ_i ≥ 0`)
- `mu9i` is non-decreasing in `θ_i`

**Theoretical relationships (complementary slackness):**
- `mu8 ≈ 1/sigma_opt`
- `mu9i[j] ≈ 1/(1 - sigma_opt * theta[j])`

**Markowitz recovery:** Setting `rho = 0, gamma = 0` should recover the standard minimum-variance solution.

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
monthly_excess = pd.read_parquet(PROC_DIR + 'monthly_excess_returns.parquet')
factors        = pd.read_parquet(PROC_DIR + 'monthly_factors.parquet')

WINDOW = 60
factor_cols_regression = ['MktRF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']

# ── Select 20 stocks with most complete data in first 60 months ──────────────
exc_w0 = monthly_excess.iloc[:WINDOW]
fac_w0 = factors[factor_cols_regression].iloc[:WINDOW]

n_obs_w0 = exc_w0.notna().sum()
top20_tickers = n_obs_w0.nlargest(20).index.tolist()

exc_toy = exc_w0[top20_tickers].copy()
print(f'Toy universe: {top20_tickers}')
print(f'Observations per stock: {exc_toy.notna().sum().values}')

In [ ]:
# ── Estimate GI parameters for toy universe ───────────────────────────────────
params_toy = estimate_gi_params(exc_toy, fac_w0, omega=0.95)

# All 20 stocks should be valid
valid = params_toy['valid_mask']
print(f'Valid stocks: {valid.sum()} / {len(valid)}')

mu0_v   = params_toy['mu0'][valid]
V0_v    = params_toy['V0'][:, valid]
D_bar_v = params_toy['D_bar'][valid]
rho_v   = params_toy['rho'][valid]
gamma_v = params_toy['gamma'][valid]
G       = params_toy['G']
F_cov   = params_toy['F_cov']

print(f'mu0 range:   [{mu0_v.min():.4f}, {mu0_v.max():.4f}]')
print(f'rho range:   [{rho_v.min():.4f}, {rho_v.max():.4f}]')
print(f'gamma range: [{gamma_v.min():.4f}, {gamma_v.max():.4f}]')

In [ ]:
# ── Solve robust SOCP (pure min-var, no return floor) ────────────────────────
result_robust = solve_gi_socp(
    mu0_v, V0_v, G, F_cov, D_bar_v, rho_v, gamma_v,
    alpha=None,   # pure minimum-variance — no worst-case return constraint
    solver='CLARABEL', verbose=False
)

print(f'Status:     {result_robust["status"]}')
print(f'Objective:  {result_robust["objective"]:.6f}  (worst-case monthly variance)')
print(f'Solve time: {result_robust["solve_time"]:.2f}s')
print(f'sigma_opt:  {result_robust["dual_vars"]["sigma_opt"]:.4f}  '
      f'(in [0, 1/theta_max={1/result_robust["theta_max"]:.1f}])')

In [ ]:
# ── Verification checks ───────────────────────────────────────────────────────
TOL = 1e-3
checks = {}

w         = result_robust['weights']
dv        = result_robust['dual_vars']
theta_arr = result_robust['theta']
sigma_opt = dv['sigma_opt']
mu9i_gi   = dv['mu9i']         # analytically computed GI dual

# Primal feasibility
checks['weights.sum ≈ 1']       = abs(w.sum() - 1.0) < TOL
checks['all weights >= -TOL']   = (w >= -TOL).all()
checks['weights not uniform']   = w.std() > 1e-4

# Dual variable signs
checks['mu7 >= 0']              = dv['mu7'] >= -TOL
checks['sigma in (0, 1/theta_max)'] = 0 < sigma_opt <= 1/result_robust['theta_max'] + TOL

# GI theoretical relationships (from complementary slackness)
if sigma_opt > 1e-10:
    mu9i_theory = 1.0 / (1.0 - sigma_opt * theta_arr)
    checks['all mu9i >= 1']    = (mu9i_gi >= 1 - TOL).all()
    max_rel = np.max(np.abs(mu9i_gi - mu9i_theory) / np.maximum(mu9i_theory, 1))
    checks['mu9i == 1/(1-σθ)'] = max_rel < 1e-10   # exact by construction
    checks['mu8 == 1/sigma']   = abs(dv['mu8'] - 1/sigma_opt) < 1e-8

# Factor model variance recovery: ||A phi||^2 should equal V0^T F_cov V0 @ phi^2
V0_v2 = V0_v[:, params_toy['valid_mask']]  # re-slice if needed
V0_sub = params_toy['V0'][:, params_toy['valid_mask']]
from numpy.linalg import eigh as _eigh
theta_sp, _, A_sp, _ = precompute_spectral(G, F_cov, V0_sub)
fac_var_A   = float(np.dot(A_sp @ w, A_sp @ w))
fac_var_VFV = float(w @ V0_sub.T @ F_cov @ V0_sub @ w)
checks['||A phi||^2 == V^T F V phi^2'] = abs(fac_var_A - fac_var_VFV) < 1e-10

print('=' * 58)
print(f'{"CHECK":<45} {"RESULT":>6}')
print('=' * 58)
all_pass = True
for name, val in checks.items():
    s = 'PASS' if val else 'FAIL'
    if not val: all_pass = False
    print(f'{name:<45} {s:>6}')
print('=' * 58)
print(f'Overall: {"ALL PASS ✓" if all_pass else "SOME FAILED"}')

print(f'\nKey values:')
print(f'  sigma_opt = {sigma_opt:.4f}   theta_max = {theta_arr[-1]:.6f}')
print(f'  mu7 = {dv["mu7"]:.6f}  mu8 = {dv["mu8"]:.4f}  (= 1/sigma = {1/sigma_opt:.4f})')
print(f'  mu9i (all should be equal ≈ {mu9i_gi[0]:.3f}): {mu9i_gi.round(3)}')
print(f'  ||A phi||^2 = {fac_var_A:.6f}   V^T F V phi^2 = {fac_var_VFV:.6f}')
print()
print('Top weights:')
for i in np.argsort(-w)[:10]:
    print(f'  {top20_tickers[i]:<8}  w={w[i]:.4f}')

In [ ]:
# ── Markowitz recovery check ──────────────────────────────────────────────────
# rho=0, gamma=0 → robust SOCP should recover min-var on the factor model cov.
# NOTE: this is NOT the same as sample-covariance Markowitz — it minimises
# phi^T (V0^T F_cov V0 + diag(D_bar)) phi (the GI factor model covariance).
result_no_unc = solve_gi_socp(
    mu0_v, V0_v, G, F_cov, D_bar_v,
    rho_valid=np.zeros_like(rho_v),
    gamma_valid=np.zeros_like(gamma_v),
    alpha=None, solver='CLARABEL'
)

# Factor-model min-var QP (direct comparison)
V0_sub  = params_toy['V0'][:, params_toy['valid_mask']]
Sigma_GI = V0_sub.T @ F_cov @ V0_sub + np.diag(D_bar_v)
n_gi = Sigma_GI.shape[0]
phi_gi = cp.Variable(n_gi, nonneg=True)
prob_gi = cp.Problem(cp.Minimize(cp.quad_form(phi_gi, Sigma_GI)), [cp.sum(phi_gi) == 1])
prob_gi.solve(solver=cp.CLARABEL)

print(f'GI SOCP (rho=gamma=0) status: {result_no_unc["status"]}')
print(f'GI SOCP objective:            {result_no_unc["objective"]:.6f}')
print(f'Factor-model QP objective:    {prob_gi.value:.6f}  ← should match')

w0 = result_no_unc['weights']
w_gi_qp = phi_gi.value
if w0 is not None and w_gi_qp is not None:
    max_diff = np.abs(w0 - w_gi_qp).max()
    corr     = np.corrcoef(w0, w_gi_qp)[0, 1]
    print(f'Max weight diff:  {max_diff:.6f}  (should be ~0)')
    print(f'Weight corr:      {corr:.6f}  (should be ~1)')
    print(f'Recovery:         {"PASS ✓" if max_diff < 0.01 else "FAIL"}')

print()
print(f'{"Ticker":<8} {"Robust (ρ=γ=0)":>18} {"Factor-model QP":>18}')
print('-' * 46)
for i, tick in enumerate(top20_tickers):
    rw = w0[i] if w0 is not None else float('nan')
    qw = w_gi_qp[i] if w_gi_qp is not None else float('nan')
    print(f'{tick:<8} {rw:>18.4f} {qw:>18.4f}')

In [ ]:
# ── Visualise: robust vs no-uncertainty weights ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(top20_tickers))
ax.bar(x - 0.2, result_robust['weights'], width=0.4, label='GI Robust (ω=0.95)', alpha=0.8)
ax.bar(x + 0.2, result_no_unc['weights'] if result_no_unc['weights'] is not None else np.zeros(20),
       width=0.4, label='No uncertainty (ρ=γ=0)', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(top20_tickers, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Portfolio weight')
ax.set_title('GI Robust vs no-uncertainty weights (20-stock toy example)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Robust weights sum: {result_robust["weights"].sum():.6f}')
print(f'No-unc weights sum: {result_no_unc["weights"].sum():.6f}')

---
## Step 5 — Benchmark Strategies

In [ ]:
from sklearn.covariance import LedoitWolf


def compute_benchmarks(
    returns_window: pd.DataFrame,
) -> dict:
    """
    Compute weights for all benchmark strategies on a given estimation window.

    OLS Markowitz uses complete-case rows only (stocks×months where all valid
    stocks have data) to ensure the sample covariance is PSD without heavy
    regularization.
    """
    min_obs = 30
    valid = returns_window.columns[returns_window.notna().sum() >= min_obs]
    n = len(valid)
    weights = {}

    # ── 1. Equal weight ───────────────────────────────────────────────────────
    weights['equal_weight'] = (valid, np.ones(n) / n)

    # ── 2. Markowitz min-variance (sample covariance) ─────────────────────────
    # Use complete-case rows to guarantee PSD sample covariance.
    try:
        R_complete = returns_window[valid].dropna()
        if len(R_complete) >= min_obs:
            Sigma_ols = R_complete.cov().values
        else:
            Sigma_ols = returns_window[valid].cov().values
        Sigma_ols = (Sigma_ols + Sigma_ols.T) / 2
        # Enforce PSD: shift by |min_eigenvalue| + small ridge
        min_eig = np.linalg.eigvalsh(Sigma_ols).min()
        if min_eig < 0:
            Sigma_ols -= min_eig * np.eye(n)
        Sigma_ols += 1e-5 * np.eye(n)
        phi_mw = cp.Variable(n, nonneg=True)
        prob = cp.Problem(cp.Minimize(cp.quad_form(phi_mw, Sigma_ols)),
                          [cp.sum(phi_mw) == 1])
        prob.solve(solver=cp.CLARABEL)
        weights['markowitz_ols'] = (valid, phi_mw.value if prob.status == 'optimal' else np.ones(n)/n)
    except Exception:
        weights['markowitz_ols'] = (valid, np.ones(n)/n)

    # ── 3. Markowitz with Ledoit-Wolf shrinkage ───────────────────────────────
    try:
        R_arr = returns_window[valid].fillna(returns_window[valid].mean()).values
        lw = LedoitWolf().fit(R_arr)
        Sigma_lw = lw.covariance_ + 1e-5 * np.eye(n)
        phi_lw = cp.Variable(n, nonneg=True)
        prob = cp.Problem(cp.Minimize(cp.quad_form(phi_lw, Sigma_lw)),
                          [cp.sum(phi_lw) == 1])
        prob.solve(solver=cp.CLARABEL)
        weights['markowitz_lw'] = (valid, phi_lw.value if prob.status == 'optimal' else np.ones(n)/n)
    except Exception:
        weights['markowitz_lw'] = (valid, np.ones(n)/n)

    # ── 4. Risk parity (1/vol) ────────────────────────────────────────────────
    vols = returns_window[valid].std().values
    vols = np.where(vols <= 0, np.nan, vols)
    inv_vol = np.where(np.isfinite(1.0 / vols), 1.0 / vols, 0.0)
    total = inv_vol.sum()
    weights['risk_parity'] = (valid, inv_vol / total if total > 0 else np.ones(n)/n)

    return weights


# Quick test on toy window
monthly_returns = pd.read_parquet(PROC_DIR + 'monthly_returns.parquet')
bm_toy = compute_benchmarks(monthly_returns.iloc[:WINDOW])
print('Benchmark weights (first 5 tickers):')
for name, (tickers, w) in bm_toy.items():
    print(f'  {name:<20}: sum={w.sum():.4f}, max={w.max():.4f}, n={len(w)}')

---
## Step 6 — Rolling Backtest Engine

In [ ]:
import pickle
import os

ESTIMATION_WINDOW = 60
RESULTS_DIR       = 'results/'
os.makedirs(RESULTS_DIR, exist_ok=True)


def run_backtest(
    monthly_returns: pd.DataFrame,
    monthly_excess: pd.DataFrame,
    factors: pd.DataFrame,
    spx_monthly_ret: pd.Series,
    omega: float = 0.95,
    alpha: Optional[float] = None,   # None = pure min-var (recommended)
    estimation_window: int = 60,
    factor_cols: list = None,
    solver: str = 'CLARABEL',
    save_path: str = None,
    verbose_every: int = 12,
) -> dict:
    """
    Rolling GI robust min-var backtest.

    alpha=None (default) → pure minimum-variance, no worst-case return floor.
    Using alpha=0.0 will frequently be infeasible because estimation uncertainty
    (gamma) typically exceeds the estimated mean returns in early windows.
    """
    if factor_cols is None:
        factor_cols = ['MktRF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']

    T = len(monthly_returns)
    dates = monthly_returns.index

    results = {
        'dates': [], 'gi_ret': [], 'ew_ret': [], 'mw_ols_ret': [],
        'mw_lw_ret': [], 'rp_ret': [], 'spx_ret': [],
        'gi_weights': [], 'gi_tickers': [], 'gi_turnover': [],
        'dual_mu2': [], 'dual_mu7': [], 'dual_mu9i': [],
        'sigma_opt': [], 'solve_status': [], 'solve_time': [], 'n_stocks': [],
    }

    prev_gi_weights = None
    prev_gi_tickers = None
    n_rebal = T - estimation_window

    print(f'Backtest: omega={omega}, alpha={alpha}, window={estimation_window}')
    print(f'Total rebalancing periods: {n_rebal}')
    print(f'From {dates[estimation_window].date()} to {dates[-2].date()}')
    print()

    t_total = time.time()

    for t_idx in range(estimation_window, T - 1):
        date_t    = dates[t_idx]
        date_next = dates[t_idx + 1]

        exc_win = monthly_excess.iloc[t_idx - estimation_window : t_idx]
        ret_win = monthly_returns.iloc[t_idx - estimation_window : t_idx]
        fac_win = factors[factor_cols].iloc[t_idx - estimation_window : t_idx]

        params = estimate_gi_params(exc_win, fac_win, omega=omega)
        valid  = params['valid_mask']
        tickers_valid = [params['ticker_names'][i] for i in range(len(valid)) if valid[i]]

        gi_result = solve_gi_socp(
            params['mu0'][valid], params['V0'][:, valid],
            params['G'], params['F_cov'],
            params['D_bar'][valid], params['rho'][valid], params['gamma'][valid],
            alpha=alpha, solver=solver
        )

        if gi_result['status'] in ['optimal', 'optimal_inaccurate'] and gi_result['weights'] is not None:
            gi_w = np.clip(gi_result['weights'], 0, None)
            gi_w /= gi_w.sum()
            dv = gi_result['dual_vars']
        else:
            gi_w = np.ones(valid.sum()) / valid.sum()
            dv = {'mu2': np.nan, 'mu7': np.nan, 'mu9i': np.full(6, np.nan), 'sigma_opt': np.nan}

        next_ret = monthly_returns.loc[date_next, tickers_valid].fillna(0).values
        gi_ret   = float(gi_w @ next_ret)

        # Turnover
        if prev_gi_weights is not None:
            curr_ser = pd.Series(gi_w, index=tickers_valid)
            prev_ser = pd.Series(prev_gi_weights, index=prev_gi_tickers)
            all_t    = list(set(tickers_valid) | set(prev_gi_tickers))
            turnover = np.abs(curr_ser.reindex(all_t).fillna(0).values -
                              prev_ser.reindex(all_t).fillna(0).values).sum()
        else:
            turnover = np.nan
        prev_gi_weights = gi_w; prev_gi_tickers = tickers_valid

        bm = compute_benchmarks(ret_win)
        def bm_ret(name):
            ticks, wb = bm[name]
            return float(wb @ monthly_returns.loc[date_next, ticks].fillna(0).values)

        results['dates'].append(date_next)
        results['gi_ret'].append(gi_ret)
        results['ew_ret'].append(bm_ret('equal_weight'))
        results['mw_ols_ret'].append(bm_ret('markowitz_ols'))
        results['mw_lw_ret'].append(bm_ret('markowitz_lw'))
        results['rp_ret'].append(bm_ret('risk_parity'))
        results['spx_ret'].append(float(spx_monthly_ret.get(date_next, np.nan)))
        results['gi_weights'].append(gi_w)
        results['gi_tickers'].append(tickers_valid)
        results['gi_turnover'].append(turnover)
        results['dual_mu2'].append(dv.get('mu2', np.nan))
        results['dual_mu7'].append(dv.get('mu7', np.nan))
        results['dual_mu9i'].append(dv.get('mu9i', np.full(6, np.nan)))
        results['sigma_opt'].append(dv.get('sigma_opt', np.nan))
        results['solve_status'].append(gi_result['status'])
        results['solve_time'].append(gi_result['solve_time'])
        results['n_stocks'].append(valid.sum())

        period_idx = t_idx - estimation_window + 1
        if period_idx % verbose_every == 0 or period_idx == 1:
            elapsed = time.time() - t_total
            rate    = period_idx / elapsed if elapsed > 0 else 1
            eta     = (n_rebal - period_idx) / rate
            print(f'[{period_idx:>4}/{n_rebal}] {date_t.date()}  '
                  f'stocks={valid.sum():3d}  {gi_result["status"]:<22}  '
                  f'{gi_result["solve_time"]:.2f}s  ETA={eta/60:.1f}min')

    elapsed = time.time() - t_total
    print(f'\nBacktest complete in {elapsed/60:.1f} min.')
    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(results, f)
        print(f'Saved to {save_path}')
    return results


print('run_backtest() defined.')

In [ ]:
# ── Runtime estimate (full universe, one period) ───────────────────────────────
monthly_returns = pd.read_parquet(PROC_DIR + 'monthly_returns.parquet')
monthly_excess  = pd.read_parquet(PROC_DIR + 'monthly_excess_returns.parquet')
factors         = pd.read_parquet(PROC_DIR + 'monthly_factors.parquet')
spx_benchmark   = pd.read_parquet(PROC_DIR + 'spx_benchmark.parquet')['SPX']

WINDOW = 60
exc_test = monthly_excess.iloc[:WINDOW]
fac_test = factors[['MktRF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']].iloc[:WINDOW]

params_test = estimate_gi_params(exc_test, fac_test, omega=0.95)
valid_test  = params_test['valid_mask']
print(f'Full universe valid stocks in period 0: {valid_test.sum()}')

t0 = time.time()
result_test = solve_gi_socp(
    params_test['mu0'][valid_test], params_test['V0'][:, valid_test],
    params_test['G'], params_test['F_cov'],
    params_test['D_bar'][valid_test], params_test['rho'][valid_test],
    params_test['gamma'][valid_test], alpha=None, solver='CLARABEL'
)
t_period = time.time() - t0

n_periods = len(monthly_returns) - WINDOW - 1
print(f'Single period solve time: {t_period:.2f}s  status: {result_test["status"]}')
print(f'Total periods: {n_periods}')
print(f'Estimated total runtime: {t_period * n_periods / 60:.1f} min')
print(f'sigma_opt: {result_test["dual_vars"]["sigma_opt"]:.3f}  '
      f'theta_max: {result_test["theta_max"]:.6f}  '
      f'1/theta_max: {1/result_test["theta_max"]:.1f}')

In [ ]:
# ── Run backtest (omega = 0.95, main result) ──────────────────────────────────
# Expected runtime: ~10–60 min depending on hardware.
# Quick test: limit to last 120 months → monthly_returns.iloc[-120:]

results_095 = run_backtest(
    monthly_returns   = monthly_returns,
    monthly_excess    = monthly_excess,
    factors           = factors,
    spx_monthly_ret   = spx_benchmark,
    omega             = 0.95,
    alpha             = None,   # pure min-var — no return floor (avoids infeasibility)
    estimation_window = WINDOW,
    solver            = 'CLARABEL',
    save_path         = RESULTS_DIR + 'backtest_omega095.pkl',
    verbose_every     = 12,
)

In [ ]:
# ── Sensitivity backtests (omega = 0.90 and 0.99) ────────────────────────────
results_090 = run_backtest(
    monthly_returns, monthly_excess, factors, spx_benchmark,
    omega=0.90, alpha=None, estimation_window=WINDOW, solver='CLARABEL',
    save_path=RESULTS_DIR + 'backtest_omega090.pkl', verbose_every=24,
)

results_099 = run_backtest(
    monthly_returns, monthly_excess, factors, spx_benchmark,
    omega=0.99, alpha=None, estimation_window=WINDOW, solver='CLARABEL',
    save_path=RESULTS_DIR + 'backtest_omega099.pkl', verbose_every=24,
)

---
## Step 7 — Performance Metrics and Dual Variable Analysis

In [ ]:
from scipy.stats import norm


def compute_metrics(returns: np.ndarray, rf: float = 0.0) -> dict:
    """
    Compute annualised performance metrics from a monthly return series.

    Parameters
    ----------
    returns : array of monthly total returns (decimal)
    rf      : annualised risk-free rate for Sharpe computation

    Returns
    -------
    dict with: mean_return, volatility, sharpe, max_drawdown, calmar,
               avg_turnover, net_ret_5bp, net_ret_20bp
    """
    returns = np.asarray(returns, dtype=float)
    valid   = ~np.isnan(returns)
    r = returns[valid]
    T = len(r)
    if T == 0:
        return {k: np.nan for k in ['mean_return', 'volatility', 'sharpe',
                                     'max_drawdown', 'calmar']}

    mean_ret = (1 + r).prod() ** (12 / T) - 1
    vol      = r.std() * np.sqrt(12)
    sharpe   = (mean_ret - rf) / vol if vol > 0 else np.nan

    cum = np.cumprod(1 + r)
    running_max = np.maximum.accumulate(cum)
    drawdowns   = (cum - running_max) / running_max
    max_dd      = drawdowns.min()
    calmar      = mean_ret / abs(max_dd) if max_dd < 0 else np.nan

    return {
        'mean_return' : mean_ret,
        'volatility'  : vol,
        'sharpe'      : sharpe,
        'max_drawdown': max_dd,
        'calmar'      : calmar,
    }


def politis_romano_se(r1: np.ndarray, r2: np.ndarray,
                      B: int = 1000, block_size: int = 6) -> tuple:
    """
    Stationary block bootstrap standard error and p-value for Sharpe ratio difference.

    Returns: (se, p_value) under null H0: SR1 - SR2 = 0
    """
    valid = ~(np.isnan(r1) | np.isnan(r2))
    r1, r2 = r1[valid], r2[valid]
    T = len(r1)
    if T < 30:
        return np.nan, np.nan

    def sharpe_diff(a, b):
        s1 = a.mean() / (a.std() + 1e-12) * np.sqrt(12)
        s2 = b.mean() / (b.std() + 1e-12) * np.sqrt(12)
        return s1 - s2

    obs_diff = sharpe_diff(r1, r2)
    boot_diffs = []
    for _ in range(B):
        # Stationary block bootstrap: geometric block lengths
        idx = []
        while len(idx) < T:
            start = np.random.randint(T)
            length = np.random.geometric(p=1/block_size)
            idx.extend(range(start, min(start + length, T)))
        idx = idx[:T]
        boot_diffs.append(sharpe_diff(r1[idx], r2[idx]))

    boot_diffs = np.array(boot_diffs)
    se = boot_diffs.std()
    # Two-sided p-value
    z  = obs_diff / (se + 1e-12)
    pval = 2 * (1 - norm.cdf(abs(z)))
    return se, pval


print('Metrics functions defined.')

In [ ]:
# ── Load backtest results ──────────────────────────────────────────────────────
import pickle

def load_results(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

results_095 = load_results(RESULTS_DIR + 'backtest_omega095.pkl')
results_090 = load_results(RESULTS_DIR + 'backtest_omega090.pkl')
results_099 = load_results(RESULTS_DIR + 'backtest_omega099.pkl')

dates = pd.to_datetime(results_095['dates'])
print(f'Results loaded.')
print(f'Out-of-sample: {dates[0].date()} → {dates[-1].date()}  ({len(dates)} months)')
print(f'Solve failures: {sum(1 for s in results_095["solve_status"] if "infeasible" in s)}/375')

In [ ]:
# ── Build performance summary table ───────────────────────────────────────────
gi095_r = np.array(results_095['gi_ret']); gi090_r = np.array(results_090['gi_ret'])
gi099_r = np.array(results_099['gi_ret']); ew_r    = np.array(results_095['ew_ret'])
ols_r   = np.array(results_095['mw_ols_ret']); lw_r = np.array(results_095['mw_lw_ret'])
rp_r    = np.array(results_095['rp_ret']);     spx_r = np.array(results_095['spx_ret'])
to_095  = np.array(results_095['gi_turnover'])

strategies = [
    ('GI Robust (ω=0.95)', gi095_r, to_095),
    ('GI Robust (ω=0.90)', gi090_r, None),
    ('GI Robust (ω=0.99)', gi099_r, None),
    ('Equal Weight',        ew_r,   None),
    ('Markowitz (OLS)',     ols_r,  None),
    ('Markowitz (LW)',      lw_r,   None),
    ('Risk Parity',         rp_r,   None),
    ('SPX Benchmark',       spx_r,  None),
]

print(f'Performance table — 1995-02 to 2026-04 ({len(dates)} months, survivorship-biased universe)')
print()
header = f'{"Strategy":<22}  {"Ann.Ret":>8}  {"Vol":>7}  {"Sharpe":>7}  {"MaxDD":>8}  {"Calmar":>7}  {"TO/yr":>7}'
print(header); print('='*len(header))

avg_to = np.nanmean(to_095) * 12
for name, rets, to in strategies:
    m  = compute_metrics(rets)
    to_s = f'{np.nanmean(to)*12:>6.0%}' if to is not None else f'{"—":>6}'
    print(f'{name:<22}  {m["mean_return"]:>7.2%}  {m["volatility"]:>6.2%}  '
          f'{m["sharpe"]:>7.3f}  {m["max_drawdown"]:>7.2%}  {m["calmar"]:>7.2f}  {to_s}')

print('='*len(header))
m_gi = compute_metrics(gi095_r)
print(f'\nGI ω=0.95 net-of-costs:  5bp → {m_gi["mean_return"]-avg_to*0.0005:.2%}/yr   '
      f'20bp → {m_gi["mean_return"]-avg_to*0.002:.2%}/yr')

In [ ]:
# ── Statistical significance (Politis-Romano bootstrap) ───────────────────────
print('Bootstrap Sharpe ratio tests: GI(ω=0.95) vs benchmarks')
print('H₀: SR_GI = SR_benchmark  [B=2000, stationary block bootstrap, block_size=6]')
print()
print(f'{"Benchmark":<22}  {"SR_GI":>7}  {"SR_BM":>7}  {"Diff":>7}  {"SE":>7}  {"p-val":>7}  {"":>5}')
print('-' * 68)

for name, rets, _ in strategies[3:]:
    valid = ~(np.isnan(gi095_r) | np.isnan(rets))
    sr_gi = gi095_r[valid].mean() / (gi095_r[valid].std() + 1e-12) * np.sqrt(12)
    sr_bm = rets[valid].mean()    / (rets[valid].std()    + 1e-12) * np.sqrt(12)
    se, pval = politis_romano_se(gi095_r, rets, B=2000)
    sig = '***' if pval < 0.01 else '** ' if pval < 0.05 else '*  ' if pval < 0.10 else '   '
    print(f'{name:<22}  {sr_gi:>7.3f}  {sr_bm:>7.3f}  {sr_gi-sr_bm:>7.3f}  {se:>7.3f}  {pval:>7.3f}  {sig}')

print()
print('Note: GI significantly UNDERPERFORMS OLS and LW (p<0.001) due to the')
print('sigma_opt = T degeneracy: the GI portfolio ignores factor structure and')
print('minimises only idiosyncratic variance, whereas OLS/LW exploit factor hedging.')
print('GI significantly OUTPERFORMS SPX (p<0.001), EW/RP not significant (p>0.22).')

In [ ]:
# ── Dual variable analysis ─────────────────────────────────────────────────────
#
# Key finding: with G = F^T M F (Gram matrix), theta_max = 1/(T-1) ≈ 1/59, so
# sigma_max = T = 59. In 97.8% of periods sigma_opt hits this cap, meaning the
# GI optimizer is at MAXIMUM uncertainty and effectively minimises ONLY
# idiosyncratic variance (C9 forces A*phi → 0 when sigma = 1/theta_max).
#
# Dual variable interpretation:
#   sigma_opt : S-procedure multiplier — how conservative the optimizer is.
#               At cap (= 59): factor structure completely ignored.
#               Below cap (early 1995): factor loadings contribute.
#   mu7       : shadow price on C7 (sigma ≤ 59). Numerically ≈ 0 when sigma
#               is at cap because relaxing C7 doesn't improve the objective
#               (all C9 already bind at A*phi = 0 when sigma = 1/theta_max).
#   mu9i      : 1/(1-sigma*theta_j) diverges when sigma*theta_max → 1.
#               Not interpretable at the cap; use sigma_opt directly.

macro_daily = pd.read_parquet(PROC_DIR + 'macro_daily.parquet')

sigma_ts = pd.Series(results_095['sigma_opt'], index=dates, name='sigma_opt')
mu7_ts   = pd.Series(results_095['dual_mu7'],  index=dates, name='mu7')
n_stocks_ts = pd.Series(results_095['n_stocks'], index=dates, name='n_stocks')

vix_monthly  = macro_daily['VIX'].resample('ME').last().reindex(dates)
move_monthly = macro_daily['MOVE'].resample('ME').last().reindex(dates)

print('sigma_opt summary (S-procedure multiplier, cap = T = 59):')
print(f'  Mean:  {sigma_ts.mean():.3f}   Std: {sigma_ts.std():.3f}')
print(f'  Min:   {sigma_ts.min():.3f}   Max: {sigma_ts.max():.3f}')
n_at_cap = (sigma_ts >= 58.99).sum()
print(f'  At cap (σ=59): {n_at_cap}/{len(sigma_ts)} periods ({n_at_cap/len(sigma_ts):.1%})')
print()
print('Periods where sigma < 59 (factor structure NOT fully suppressed):')
below_cap = sigma_ts[sigma_ts < 58.99]
print(below_cap.round(3).to_string())
print()
print('Correlations of sigma_opt with macro:')
print(f'  corr(sigma_opt, VIX):  {sigma_ts.corr(vix_monthly):.3f}')
print(f'  corr(sigma_opt, MOVE): {sigma_ts.corr(move_monthly):.3f}')
print()

# Weight concentration (effective N)
hhi = pd.Series([(w**2).sum() for w in results_095['gi_weights']], index=dates)
print('Portfolio concentration (HHI = Σw²):')
print(f'  Mean effective N (1/HHI): {(1/hhi).mean():.1f}  '
      f'(range: {(1/hhi).min():.0f}–{(1/hhi).max():.0f})')
print()

# Turnover
to_arr = np.array(results_095['gi_turnover'])
print(f'Avg one-way turnover: {np.nanmean(to_arr)*12:.1%}/yr')
print(f'  Net-of-5bp:  {compute_metrics(np.array(results_095["gi_ret"]))["ret"] - np.nanmean(to_arr)*12*0.0005:.2%}/yr')
print(f'  Net-of-20bp: {compute_metrics(np.array(results_095["gi_ret"]))["ret"] - np.nanmean(to_arr)*12*0.002:.2%}/yr')

In [ ]:
# ── sigma_opt time series plot ────────────────────────────────────────────────
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(sigma_ts.index, sigma_ts.values, color='navy', lw=1,
         label='σ* (S-procedure multiplier)')
ax1.axhline(59, color='red', ls='--', lw=1, alpha=0.7, label='σ_max = T = 59')
ax1_r = ax1.twinx()
ax1_r.fill_between(vix_monthly.index, vix_monthly.values, alpha=0.2, color='orange')
ax1_r.set_ylabel('VIX', color='orange')
ax1.set_ylabel('σ*'); ax1.set_ylim(48, 62); ax1.legend(fontsize=9)
ax1.set_title('S-procedure multiplier σ* vs VIX\n'
               '(σ*≈T in 97.8% of periods: GI operates at max uncertainty, '
               'factor model fully suppressed)')

ax2.plot(n_stocks_ts.index, n_stocks_ts.values, color='steelblue', lw=1,
         label='Stocks in GI universe')
eff_n = pd.Series([(1/w**2).sum()**-1 if (w**2).sum() > 0 else np.nan
                    for w in results_095['gi_weights']], index=dates)
# actually 1/HHI for effective n
hhi_s = pd.Series([(w**2).sum() for w in results_095['gi_weights']], index=dates)
ax2.plot(hhi_s.index, 1/hhi_s.values, color='orange', lw=1, label='Effective N (1/HHI)')
ax2.set_ylabel('Number of stocks'); ax2.legend(fontsize=9)
ax2.set_title('Portfolio size vs effective diversification')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

print(f'\nInterpreting sigma_opt = {sigma_ts.mean():.2f} ≈ T = 59:')
print('  When sigma = 1/theta_max = T, the C9 constraints force (A phi)_j = 0 for all j.')
print('  This means the portfolio has ZERO exposure to factor uncertainty — it')
print('  minimises idiosyncratic variance only (the delta term in the objective).')
print('  The GI robust portfolio with T=60, m=6 degenerates to min-idiosyncratic-var')
print('  in nearly every period, which explains the very low realised vol (6.63%).')

In [ ]:
# ── Plots: cumulative returns, sigma_opt, and risk-return scatter ─────────────
import os

fig_dir = RESULTS_DIR
figs = ['fig1_cumulative_returns.png', 'fig2_sigma_opt_concentration.png',
        'fig3_rolling_sharpe.png',     'fig4_risk_return.png']

print('Saved figures:')
for f in figs:
    path = fig_dir + f
    if os.path.exists(path):
        size = os.path.getsize(path) // 1024
        print(f'  {f}  ({size} KB)')
    else:
        print(f'  {f}  NOT FOUND — run the standalone script to regenerate')

# Display inline (Jupyter)
from IPython.display import Image, display
for f in figs:
    path = fig_dir + f
    if os.path.exists(path):
        print(f'\n--- {f} ---')
        display(Image(filename=path))

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=' * 90)
print('BACKTEST COMPLETE — GI Robust Portfolio Optimization Empirical Results')
print('=' * 90)
print()
print(f'Out-of-sample: {dates[0].date()} → {dates[-1].date()}  ({len(dates)} months)')
print(f'Solve failures: 0/375   All three ω variants complete.')
print()
print('PERFORMANCE SUMMARY (all figures inflated by survivorship bias)')
print('-' * 70)
rows = [
    ('GI Robust ω=0.95', '16.42%', '6.63%', '2.478', '-9.08%',  '1.81'),
    ('GI Robust ω=0.90', '16.48%', '6.52%', '2.528', '-9.01%',  '1.83'),
    ('GI Robust ω=0.99', '16.24%', '6.84%', '2.375', '-9.18%',  '1.77'),
    ('Equal Weight',      '18.92%', '8.79%', '2.153', '-19.67%', '0.96'),
    ('Markowitz (OLS)',   '19.84%', '4.10%', '4.835', '-4.99%',  '3.98'),
    ('Markowitz (LW)',    '19.81%', '3.65%', '5.428', '-3.68%',  '5.39'),
    ('Risk Parity',       '18.45%', '8.55%', '2.158', '-19.66%', '0.94'),
    ('SPX Benchmark',     '11.15%','15.13%', '0.737', '-50.91%', '0.22'),
]
print(f'  {"Strategy":<22}  {"Ret":>7}  {"Vol":>7}  {"Sharpe":>7}  {"MaxDD":>8}  {"Calmar":>7}')
for r in rows:
    print(f'  {r[0]:<22}  {r[1]:>7}  {r[2]:>7}  {r[3]:>7}  {r[4]:>8}  {r[5]:>7}')
print()
print('MAIN DUAL VARIABLE FINDING')
print('-' * 70)
print('  With G = F^T M F (Gram matrix), theta_max = 1/(T-1) = 1/59,')
print('  sigma_max = T-1 = 59. The S-procedure multiplier σ* hits this cap')
print('  in 97.9% of periods (367/375). At σ* = 1/theta_max:')
print('    • C9 forces (A*phi)_j = 0 for all j  →  zero factor exposure')
print('    • GI reduces to min-idiosyncratic-variance portfolio')
print('    • mu9i = 1/(1-σ*θ) diverges (not interpretable in this regime)')
print('    • mu7 ≈ 0 (relaxing C7 has no value once A*phi=0)')
print()
print('  Consequence: GI significantly UNDERPERFORMS OLS/LW Markowitz')
print('  (SR diff ≈ -2.1/–2.7, p<0.001) because it cannot exploit factor hedging.')
print('  GI significantly OUTPERFORMS SPX (SR diff +1.56, p<0.001).')
print()
print('CAVEATS')
print('-' * 70)
print('  1. Markowitz (OLS/LW) vol (4.10%/3.65%) very low — consistent with')
print('     min-var concentration in the biased universe. Not representative.')
print('  2. The sigma_opt = T degeneracy is structural: occurs regardless of ω or')
print('     window length because G=Gram makes all theta_j = 1/(T-1).')